# Procesamiento Avanzado de Datos con PySpark
Este notebook incluye ejemplos extensivos de:
- Transformaciones y Acciones (5 ejemplos de cada uno).
- Uso de UDFs personalizadas (5 ejemplos).
- Funciones incorporadas de Spark (5 ejemplos).
- Consultas con Spark SQL.
- Construcción de un ETL avanzado con join y guardado en formato Parquet.

In [1]:

import findspark
import pandas
findspark.init()
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType
)

jars = [
    "/var/snp-dwh/spark_jobs/jars/spark-excel_2.12-3.5.1_0.20.4.jar",
    "/var/snp-dwh/spark_jobs/jars/poi-4.1.2.jar",
    "/var/snp-dwh/spark_jobs/jars/poi-ooxml-4.1.2.jar",
    "/var/snp-dwh/spark_jobs/jars/xmlbeans-3.1.0.jar",
    "/var/snp-dwh/spark_jobs/jars/commons-math3-3.6.1.jar",
    "/var/snp-dwh/spark_jobs/jars/ooxml-schemas-1.4.jar",
    "/var/snp-dwh/spark_jobs/jars/postgresql-42.7.3.jar"
]

spark = SparkSession.builder \
    .appName("ExcelFallback") \
    .config("spark.master", "local[*]") \
    .config("spark.jars", ",".join(jars)) \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()
spark.sparkContext.setCheckpointDir('/var/snp-dwh/localtest/cheakpoint/')
# Definición del schema exacto que me pasaste

schema = StructType([
    StructField("codigo_", StringType(), True),
    StructField("TIPO", StringType(), True),
    StructField("EJE", StringType(), True),
    StructField("NOMBRE_DEL_OBJETIVO", StringType(), True),
    StructField("NOMBRE_DE_LA_POLITICA", StringType(), True),
    StructField("META", StringType(), True),
    StructField("INDICADOR", StringType(), True),
    StructField("FUENTE_DE_INFORMACION", StringType(), True),
    StructField("GRUPO_DE_DESAGREGACION", StringType(), True),
    StructField("NIVEL_DE_DESAGREGACION", StringType(), True),
    StructField("CODIGO_GEOGRAFICO_DPA", StringType(), True),
    StructField("MES_ANIO", StringType(), True),
    StructField("FECHA", DateType(), True),
    StructField("ESTIMADOR", DoubleType(), True),
    StructField("ERROR_ESTANDAR", DoubleType(), True),
    StructField("LIMITE_INFERIOR", DoubleType(), True),
    StructField("LIMITE_SUPERIOR", DoubleType(), True),
    StructField("COEFICIENTE_DE_VARIACION", DoubleType(), True),
    StructField("NUMERADOR", DoubleType(), True),
    StructField("DENOMINADOR", DoubleType(), True),
    StructField("NOMBRE_DEL_EJE", StringType(), True),
    StructField("PERIODICIDAD_FICHA_METODOLOGICA", StringType(), True),
    StructField("FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA", StringType(), True),
    StructField("DESAGREGACION_FICHA_METODOLOGICA", StringType(), True),
    StructField("PERIODICIDAD_DEL_DATO", StringType(), True)
])



# Ruta en HDFS (ajusta si cambias el archivo)
hdfs_path = "hdfs://192.168.1.144:8020/data/datos_pnd2425-13-05-2025_n.xlsx"

# Leer usando dataAddress para empezar desde la fila 3
df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("dataAddress", "'Indicadores PND24-25'!A3") \
    .option("maxRowsInMemory", 500) \
    .schema(schema) \
    .load(hdfs_path)


25/08/04 17:12:38 WARN Utils: Your hostname, testnode1 resolves to a loopback address: 127.0.1.1; using 192.168.1.144 instead (on interface eth0)
25/08/04 17:12:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/08/04 17:12:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/04 17:12:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/08/04 17:12:39 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
spark.stop()

In [39]:
df_expr4 = df.selectExpr(
    "codigo_",
    "EJE",
    "CASE WHEN ESTIMADOR > 0.12 THEN 'ALTO' ELSE 'BAJO' END as CLASE_ESTIMADOR"
)
df_expr4.show(3)

+-------+------+---------------+
|codigo_|   EJE|CLASE_ESTIMADOR|
+-------+------+---------------+
|  1.1.1|SOCIAL|           ALTO|
|  1.1.1|SOCIAL|           BAJO|
|  1.1.1|SOCIAL|           BAJO|
+-------+------+---------------+
only showing top 3 rows



In [40]:
df_expr5 = df.selectExpr("concat(codigo_, '-', EJE) as CODIGO_FULL", "ESTIMADOR")
df_expr5.show(3)

+------------+---------+
| CODIGO_FULL|ESTIMADOR|
+------------+---------+
|1.1.1-SOCIAL|    0.131|
|1.1.1-SOCIAL|   0.1161|
|1.1.1-SOCIAL|   0.1118|
+------------+---------+
only showing top 3 rows



In [41]:
df_expr10 = df.selectExpr(
    "codigo_",
    "EJE",
    "ESTIMADOR",
    "ESTIMADOR * 100 as ESTIMADOR_PCT",
    "sqrt(ESTIMADOR) as ESTIMADOR_SQRT",
    "CASE WHEN ESTIMADOR > 0.12 THEN 'OK' ELSE 'REVISION' END as STATUS"
)
df_expr10.show(3)

+-------+------+---------+------------------+-------------------+--------+
|codigo_|   EJE|ESTIMADOR|     ESTIMADOR_PCT|     ESTIMADOR_SQRT|  STATUS|
+-------+------+---------+------------------+-------------------+--------+
|  1.1.1|SOCIAL|    0.131|13.100000000000001| 0.3619392214170772|      OK|
|  1.1.1|SOCIAL|   0.1161|             11.61| 0.3407345007480164|REVISION|
|  1.1.1|SOCIAL|   0.1118|             11.18|0.33436506994600973|REVISION|
+-------+------+---------+------------------+-------------------+--------+
only showing top 3 rows



## 1. Ejemplos de Transformaciones (5 ejemplos)

In [30]:
# 1) Seleccionar columnas
df1 = df.select("EJE", "ESTIMADOR")


In [ ]:

# 1) Seleccionar columnas
df1 = df.select("EJE", "ESTIMADOR")

# 2) Filtrar registros
df2 = df.filter(F.col("ESTIMADOR") > 50)

# 3) Agregar nueva columna
df3 = df.withColumn("ESTIMADOR_X2", F.col("ESTIMADOR")*2)

# 4) Agrupar y calcular promedio
df4 = df.groupBy("EJE").agg(F.avg("ESTIMADOR").alias("promedio"))
df4.selectExpr( '*', 'count(*)' )
# 5) Ordenar por ESTIMADOR descendente
df5 = df.orderBy(F.col("ESTIMADOR").desc())


In [38]:
df4.selectExpr("count(*) as total_filas").show()


+-----------+
|total_filas|
+-----------+
|          6|
+-----------+



In [35]:
df4.select('promedio').show()

+------------------+
|          promedio|
+------------------+
|               NaN|
|27248.961465986922|
| 3881953.126252223|
| 1386303.902823752|
| 25.54663879598662|
| 42.73963771268122|
+------------------+



## 2. Ejemplos de Acciones (5 ejemplos)

In [ ]:

# 1) show()
df.show(3)


+-------+----+------+--------------------+---------------------+--------------------+--------------------+---------------------+----------------------+----------------------+---------------------+--------+----------+---------+--------------+---------------+---------------+------------------------+---------+-----------+--------------+-------------------------------+-----------------------------------------+--------------------------------+---------------------+
|codigo_|TIPO|   EJE| NOMBRE_DEL_OBJETIVO|NOMBRE_DE_LA_POLITICA|                META|           INDICADOR|FUENTE_DE_INFORMACION|GRUPO_DE_DESAGREGACION|NIVEL_DE_DESAGREGACION|CODIGO_GEOGRAFICO_DPA|MES_ANIO|     FECHA|ESTIMADOR|ERROR_ESTANDAR|LIMITE_INFERIOR|LIMITE_SUPERIOR|COEFICIENTE_DE_VARIACION|NUMERADOR|DENOMINADOR|NOMBRE_DEL_EJE|PERIODICIDAD_FICHA_METODOLOGICA|FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA|DESAGREGACION_FICHA_METODOLOGICA|PERIODICIDAD_DEL_DATO|
+-------+----+------+--------------------+---------------------+------

In [ ]:

# 1) show()
df.show(3)

# 2) count()
print("Cantidad de registros:", df.count())

# 3) collect()
datos = df.select("EJE").limit(5).collect()
print(datos)

# 4) first()
print("Primer registro:", df.first())

# 5) take()
print("Primeros 2 registros:", df.take(2))


+-------+----+------+--------------------+---------------------+--------------------+--------------------+---------------------+----------------------+----------------------+---------------------+--------+----------+---------+--------------+---------------+---------------+------------------------+---------+-----------+--------------+-------------------------------+-----------------------------------------+--------------------------------+---------------------+
|codigo_|TIPO|   EJE| NOMBRE_DEL_OBJETIVO|NOMBRE_DE_LA_POLITICA|                META|           INDICADOR|FUENTE_DE_INFORMACION|GRUPO_DE_DESAGREGACION|NIVEL_DE_DESAGREGACION|CODIGO_GEOGRAFICO_DPA|MES_ANIO|     FECHA|ESTIMADOR|ERROR_ESTANDAR|LIMITE_INFERIOR|LIMITE_SUPERIOR|COEFICIENTE_DE_VARIACION|NUMERADOR|DENOMINADOR|NOMBRE_DEL_EJE|PERIODICIDAD_FICHA_METODOLOGICA|FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA|DESAGREGACION_FICHA_METODOLOGICA|PERIODICIDAD_DEL_DATO|
+-------+----+------+--------------------+---------------------+------

## 3. UDFs Personalizadas (5 ejemplos)

In [ ]:

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, DoubleType

# 1) Clasificar estimador
def clasificar(valor):
    if valor is None: return "NA"
    return "ALTO" if valor > 100 else "BAJO"
udf_clasificar = udf(clasificar, StringType())

# 2) Extraer inicial de EJE
udf_inicial = udf(lambda x: x[0] if x else "?", StringType())

# 3) Calcular raíz cuadrada
import math
udf_sqrt = udf(lambda x: math.sqrt(x) if x is not None and x >= 0 else None, DoubleType())

# 4) Convertir a mayúsculas
udf_upper = udf(lambda x: x.upper() if x else None, StringType())

# 5) Concatenar campos
udf_concat = udf(lambda a,b: f"{a}-{b}" if a and b else None, StringType())

df_udf = df.withColumn("CLASE", udf_clasificar("ESTIMADOR")) \
    .withColumn("INICIAL_EJE", udf_inicial("EJE")) \
        .withColumn("SQRT_ESTIMADOR", udf_sqrt("ESTIMADOR")) \
            .withColumn("OBJETIVO_MAYUS", udf_upper("NOMBRE_DEL_OBJETIVO")) \
                .withColumn("CODIGO_FULL", udf_concat("codigo_", "EJE"))

df_udf.select("ESTIMADOR","CLASE","INICIAL_EJE","SQRT_ESTIMADOR","CODIGO_FULL").show(3)


+---------+-----+-----------+-------------------+------------+
|ESTIMADOR|CLASE|INICIAL_EJE|     SQRT_ESTIMADOR| CODIGO_FULL|
+---------+-----+-----------+-------------------+------------+
|    0.131| BAJO|          S| 0.3619392214170772|1.1.1-SOCIAL|
|   0.1161| BAJO|          S| 0.3407345007480164|1.1.1-SOCIAL|
|   0.1118| BAJO|          S|0.33436506994600973|1.1.1-SOCIAL|
+---------+-----+-----------+-------------------+------------+
only showing top 3 rows



## 4. Funciones Incorporadas (5 ejemplos)

In [ ]:

df_func = df.withColumn("LOG_ESTIMADOR", F.log(F.col("ESTIMADOR"))) \
    .withColumn("ANIO", F.year("FECHA"))\
        .withColumn("MES", F.month("FECHA")) \
            .withColumn("TRIM", F.trim("TIPO")) \
                .withColumn("SUBSTR_OBJ", F.substring("NOMBRE_DEL_OBJETIVO", 1, 10))

df_func.select("ESTIMADOR","LOG_ESTIMADOR","ANIO","MES","SUBSTR_OBJ").show(3)


+---------+-------------------+----+---+----------+
|ESTIMADOR|      LOG_ESTIMADOR|ANIO|MES|SUBSTR_OBJ|
+---------+-------------------+----+---+----------+
|    0.131|-2.0325579557809856|2010|  1|1. Mejorar|
|   0.1161| -2.153303390278291|2011|  1|1. Mejorar|
|   0.1118| -2.191043718261138|2012|  1|1. Mejorar|
+---------+-------------------+----+---+----------+
only showing top 3 rows



## 5. Join con ID Genérico

In [ ]:

# Crear un DataFrame auxiliar para el join
df_aux = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("dataAddress", "'Indicadores PND24-25'!A3") \
    .option("maxRowsInMemory", 500) \
    .schema(schema) \
    .load(hdfs_path)
# Inner Join
# Renombrar todas las columnas de df_aux excepto 'codigo_'
df_aux_renamed = df_aux
for c in df_aux.columns:
    if c != "codigo_":   # no renombramos la clave del join
        df_aux_renamed = df_aux_renamed.withColumnRenamed(c, f"{c}_AUX")
df_aux_renamed= df_aux_renamed.checkpoint()
# ✅ Join sin ambigüedad

df_join = df_udf.join(df_aux_renamed, on="codigo_", how="inner")


In [ ]:
df_join.columns

['codigo_',
 'TIPO',
 'EJE',
 'NOMBRE_DEL_OBJETIVO',
 'NOMBRE_DE_LA_POLITICA',
 'META',
 'INDICADOR',
 'FUENTE_DE_INFORMACION',
 'GRUPO_DE_DESAGREGACION',
 'NIVEL_DE_DESAGREGACION',
 'CODIGO_GEOGRAFICO_DPA',
 'MES_ANIO',
 'FECHA',
 'ESTIMADOR',
 'ERROR_ESTANDAR',
 'LIMITE_INFERIOR',
 'LIMITE_SUPERIOR',
 'COEFICIENTE_DE_VARIACION',
 'NUMERADOR',
 'DENOMINADOR',
 'NOMBRE_DEL_EJE',
 'PERIODICIDAD_FICHA_METODOLOGICA',
 'FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA',
 'DESAGREGACION_FICHA_METODOLOGICA',
 'PERIODICIDAD_DEL_DATO',
 'CLASE',
 'INICIAL_EJE',
 'SQRT_ESTIMADOR',
 'OBJETIVO_MAYUS',
 'CODIGO_FULL',
 'TIPO_AUX',
 'EJE_AUX',
 'NOMBRE_DEL_OBJETIVO_AUX',
 'NOMBRE_DE_LA_POLITICA_AUX',
 'META_AUX',
 'INDICADOR_AUX',
 'FUENTE_DE_INFORMACION_AUX',
 'GRUPO_DE_DESAGREGACION_AUX',
 'NIVEL_DE_DESAGREGACION_AUX',
 'CODIGO_GEOGRAFICO_DPA_AUX',
 'MES_ANIO_AUX',
 'FECHA_AUX',
 'ESTIMADOR_AUX',
 'ERROR_ESTANDAR_AUX',
 'LIMITE_INFERIOR_AUX',
 'LIMITE_SUPERIOR_AUX',
 'COEFICIENTE_DE_VARIACION_AUX',
 'NU

## 6. ETL Avanzado y Guardado en Parquet

In [ ]:
df_join.select('codigo_').count()

83523903

In [ ]:
df_etl = df_join.filter(F.col("ESTIMADOR").isNotNull()) \
                .withColumn("ANIO", F.year("FECHA")) \
                .dropDuplicates(["codigo_", "FECHA"])

output_path = "hdfs://192.168.1.144:8020/data/salida_indicadores_pnd_extenso.parquet"
output_path = "/var/snp-dwh/sftp/output/habitantes_procesado.parquet"

df_etl.write.mode("overwrite").parquet(output_path)


In [ ]:
#df_etl.write.mode("overwrite").parquet(output_path)
df1 = spark.read.parquet("/var/snp-dwh/sftp/output/habitantes_procesado.parquet")

In [ ]:
df1.count()

1361

In [ ]:
spark.stop()